# 03 - EDA Trader Performance
Analyze PnL, win rates, leverage, direction bias, time patterns, and symbol profitability.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

project_root = Path('..').resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.data_loader import load_raw_datasets
from src.preprocessing import preprocess_fear_greed, preprocess_trades, merge_datasets

In [ ]:
project_root = Path('..').resolve()
fig_dir = project_root / 'outputs' / 'figures'
fig_dir.mkdir(parents=True, exist_ok=True)

fg_raw, trades_raw = load_raw_datasets(
    fear_greed_path=project_root / 'data' / 'raw' / 'fear_greed_index.csv',
    trades_path=project_root / 'data' / 'raw' / 'hyperliquid_trades.csv'
)
fg, _ = preprocess_fear_greed(fg_raw)
trades, _ = preprocess_trades(trades_raw)
merged = merge_datasets(trades, fg)
merged['date'] = pd.to_datetime(merged['date'])

In [ ]:
# 1) PnL distribution histogram (symlog for heavy tails)
plt.figure(figsize=(11,5))
sns.histplot(merged['closedPnL'].dropna(), bins=100)
plt.xscale('symlog')
mean_pnl = merged['closedPnL'].mean()
median_pnl = merged['closedPnL'].median()
plt.axvline(mean_pnl, color='red', linestyle='--', label=f'Mean={mean_pnl:.2f}')
plt.axvline(median_pnl, color='green', linestyle='--', label=f'Median={median_pnl:.2f}')
plt.title('Closed PnL Distribution (symlog)')
plt.xlabel('Closed PnL')
plt.ylabel('Trade Count')
plt.legend()
plt.annotate('Insight: PnL is heavy-tailed; robust stats are preferred.', xy=(0.01, 0.02), xycoords='axes fraction')
plt.tight_layout()
plt.savefig(fig_dir / 'nb3_pnl_distribution.png', dpi=300)
plt.show()

In [ ]:
# 2) Win rate by trader (top/bottom 20, min 10 trades)
trader_stats = merged.groupby('account', as_index=False).agg(trades=('closedPnL','size'), win_rate=('is_profitable','mean'))
eligible = trader_stats[trader_stats['trades'] >= 10].copy()
top20 = eligible.sort_values('win_rate', ascending=False).head(20)
bot20 = eligible.sort_values('win_rate', ascending=True).head(20)
display(top20[['account','trades','win_rate']])
display(bot20[['account','trades','win_rate']])

In [ ]:
# 3) Daily aggregate PnL area chart
daily = merged.groupby('date', as_index=False).agg(daily_total_pnl=('closedPnL','sum'))
plt.figure(figsize=(12,5))
plt.fill_between(daily['date'], daily['daily_total_pnl'], alpha=0.4)
plt.plot(daily['date'], daily['daily_total_pnl'])
plt.title('Daily Aggregate PnL')
plt.xlabel('Date')
plt.ylabel('Total PnL')
plt.annotate('Insight: aggregate outcomes are regime-sensitive.', xy=(0.01, 0.02), xycoords='axes fraction')
plt.tight_layout()
plt.savefig(fig_dir / 'nb3_daily_pnl_area.png', dpi=300)
plt.show()

In [ ]:
# 4) Leverage usage distribution and boxplot
fig, axes = plt.subplots(1,2, figsize=(14,5))
sns.histplot(merged['leverage'].dropna(), bins=50, ax=axes[0])
axes[0].set_title('Leverage Histogram')
axes[0].set_xlabel('Leverage')
axes[0].set_ylabel('Trades')
sns.boxplot(y=merged['leverage'].dropna(), ax=axes[1])
axes[1].set_title('Leverage Box Plot')
axes[1].set_ylabel('Leverage')
axes[0].text(0.01, 0.93, 'Insight: leverage is right-skewed.', transform=axes[0].transAxes)
plt.tight_layout()
plt.savefig(fig_dir / 'nb3_leverage_usage.png', dpi=300)
plt.show()

In [ ]:
# 5) Long vs short ratio overall and by month
overall_ratio = merged['side'].value_counts(normalize=True).rename('ratio')
by_month = merged.groupby(['month','side']).size().unstack(fill_value=0)
by_month = by_month.div(by_month.sum(axis=1), axis=0)
display(overall_ratio)
display(by_month.tail(12))

In [ ]:
# 6) Trade volume by hour and day-of-week
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
pivot = merged.pivot_table(index='day_of_week', columns='hour', values='account', aggfunc='count').reindex(day_order)
plt.figure(figsize=(14,5))
sns.heatmap(pivot, cmap='magma')
plt.title('Trade Volume Heatmap: Day of Week vs Hour')
plt.xlabel('Hour of Day')
plt.ylabel('Day of Week')
plt.annotate('Insight: trading activity clusters in specific intraday windows.', xy=(0.01, 0.02), xycoords='axes fraction')
plt.tight_layout()
plt.savefig(fig_dir / 'nb3_volume_hour_dow_heatmap.png', dpi=300)
plt.show()

In [ ]:
# 7) Symbol traded most and most profitable
symbol_stats = merged.groupby('symbol', as_index=False).agg(
    trade_count=('closedPnL','size'),
    total_pnl=('closedPnL','sum'),
    avg_pnl=('closedPnL','mean')
)
display(symbol_stats.sort_values('trade_count', ascending=False).head(15))
display(symbol_stats.sort_values('total_pnl', ascending=False).head(15))